# Day 5 — PySpark: All Joins

> **Topics:** INNER · LEFT · RIGHT · FULL OUTER · SELF · CROSS  
> **Run order:** top to bottom — setup cell creates all DataFrames

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day5_Joins') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# ── customers (customer_id=4 Dave has no orders) ──────────────────────────────
customers_data = [
    (1, 'Alice', 'alice@example.com', 'New York'),
    (2, 'Bob',   'bob@example.com',   'Chicago'),
    (3, 'Carol', 'carol@example.com', 'Houston'),
    (4, 'Dave',  'dave@example.com',  'Phoenix'),
    (5, 'Eve',   'eve@example.com',   'Seattle'),
]
customers_schema = StructType([
    StructField('customer_id', IntegerType(), False),
    StructField('name',        StringType(),  True),
    StructField('email',       StringType(),  True),
    StructField('city',        StringType(),  True),
])
df_customers = spark.createDataFrame(customers_data, schema=customers_schema)

# ── orders (customer_id=99 is orphan — no matching customer) ─────────────────
orders_data = [
    (1,  1,  250.00, 'completed', '2024-01-05'),
    (2,  1,  125.50, 'pending',   '2024-01-12'),
    (3,  2,   89.99, 'completed', '2024-01-07'),
    (4,  3,  450.00, 'completed', '2024-01-10'),
    (5,  3,  210.00, 'cancelled', '2024-01-14'),
    (6,  5,  320.00, 'completed', '2024-01-08'),
    (7, 99,   75.00, 'pending',   '2024-01-15'),   # orphan
]
orders_schema = StructType([
    StructField('order_id',    IntegerType(), False),
    StructField('customer_id', IntegerType(), True),
    StructField('amount',      DoubleType(),  True),
    StructField('status',      StringType(),  True),
    StructField('order_date',  StringType(),  True),
])
df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# ── employees (self-join hierarchy) ──────────────────────────────────────────
employees_data = [
    (1, 'Sarah', 'CEO',            None),
    (2, 'Tom',   'VP Engineering',    1),
    (3, 'Priya', 'VP Marketing',      1),
    (4, 'Alice', 'Engineer',          2),
    (5, 'Bob',   'Engineer',          2),
    (6, 'Carol', 'Designer',          3),
    (7, 'Dave',  'Analyst',           3),
]
employees_schema = StructType([
    StructField('emp_id',     IntegerType(), False),
    StructField('name',       StringType(),  True),
    StructField('role',       StringType(),  True),
    StructField('manager_id', IntegerType(), True),
])
df_employees = spark.createDataFrame(employees_data, schema=employees_schema)

# ── products (for CROSS JOIN) ─────────────────────────────────────────────────
products_data = [
    (1, 'Widget A',  'Hardware',  29.99),
    (2, 'Widget B',  'Hardware',  49.99),
    (3, 'Service X', 'Software',  99.00),
    (4, 'Service Y', 'Software', 149.00),
]
products_schema = StructType([
    StructField('product_id',   IntegerType(), False),
    StructField('product_name', StringType(),  True),
    StructField('category',     StringType(),  True),
    StructField('price',        DoubleType(),  True),
])
df_products = spark.createDataFrame(products_data, schema=products_schema)

print('customers:', df_customers.count())
print('orders:   ', df_orders.count())
print('employees:', df_employees.count())
print('products: ', df_products.count())

---
## Preview the DataFrames

In [ ]:
print('── customers ──')
df_customers.show()
print('── orders ──')
df_orders.show()
print('── employees ──')
df_employees.show()
print('── products ──')
df_products.show()

---
## 1. INNER JOIN — Only Matching Rows

- Dave (customer_id=4, no orders) → **dropped**
- Orphan order (customer_id=99) → **dropped**

In [ ]:
# Basic INNER JOIN — string 'on' deduplicates the key column
df_inner = df_orders.join(df_customers, on='customer_id', how='inner')
df_inner.printSchema()

In [ ]:
# Select the columns we want
df_inner = (
    df_orders
    .join(df_customers, on='customer_id', how='inner')
    .select('order_id', 'name', 'city', 'amount', 'status')
    .orderBy('order_id')
)
print('INNER JOIN — 6 rows (orphan order_id=7 excluded, Dave has no orders):')
df_inner.show()

In [ ]:
# Column expression join — use when column names differ between DataFrames
df_inner2 = (
    df_orders
    .join(
        df_customers,
        df_orders.customer_id == df_customers.customer_id,   # Column expression
        how='inner'
    )
    # With column expression, BOTH customer_id columns are kept — must drop one
    .drop(df_customers.customer_id)
    .select('order_id', 'name', 'city', 'amount')
    .orderBy('order_id')
)
df_inner2.show()

---
## 2. LEFT JOIN — All Left Rows + Matched Right

In [ ]:
# All customers — Dave appears with NULL order columns
df_left = (
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .select('customer_id', 'name', 'order_id', 'amount')
    .orderBy('customer_id', 'order_id')
)
print('LEFT JOIN — Dave row has null order_id / amount:')
df_left.show()

In [ ]:
# Find customers with NO orders — LEFT + isNull()
# Never use == None in PySpark — use .isNull()
df_no_orders = (
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .filter(F.col('order_id').isNull())
    .select('customer_id', 'name', 'email')
)
print('Customers with no orders:')
df_no_orders.show()

In [ ]:
# LEFT JOIN + GROUP BY — total orders + revenue per customer (including zeros)
# F.count('order_id') counts non-NULL → 0 for Dave
# F.coalesce(F.sum('amount'), F.lit(0)) → 0.0 for Dave instead of NULL
df_revenue = (
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .groupBy('customer_id', 'name')
    .agg(
        F.count('order_id').alias('total_orders'),
        F.coalesce(F.sum('amount'), F.lit(0)).alias('total_revenue'),
    )
    .orderBy('total_revenue', ascending=False)
)
print('Revenue per customer — Dave shows 0:')
df_revenue.show()

---
## 3. RIGHT JOIN — All Right Rows + Matched Left

In [ ]:
# All orders — orphan order (customer_id=99) appears with NULL customer columns
df_right = (
    df_customers
    .join(df_orders, on='customer_id', how='right')
    .select('name', 'city', 'order_id', 'customer_id', 'amount')
    .orderBy('order_id')
)
print('RIGHT JOIN — order_id=7 (cust 99) has null customer name/city:')
df_right.show()

---
## 4. SELF JOIN — Table Joined to Itself

Must `.alias()` the DataFrame twice — then use `F.col('alias.column')` to reference columns.

In [ ]:
df_employees.show()

In [ ]:
# SELF JOIN: employee → manager
# Alias the same DataFrame as 'e' (employee) and 'm' (manager)
emp = df_employees.alias('e')
mgr = df_employees.alias('m')

df_hierarchy = (
    emp.join(
        mgr,
        F.col('e.manager_id') == F.col('m.emp_id'),
        how='left'       # LEFT so CEO (manager_id=None) still appears
    )
    .select(
        F.col('e.emp_id'),
        F.col('e.name').alias('employee'),
        F.col('e.role'),
        F.col('m.name').alias('manager'),
        F.col('m.role').alias('manager_role'),
    )
    .orderBy('e.emp_id')
)
print('SELF JOIN — Sarah (CEO) has null manager:')
df_hierarchy.show()

In [ ]:
# SELF JOIN: who reports directly to 'Tom'?
df_toms_team = (
    emp.join(
        mgr,
        F.col('e.manager_id') == F.col('m.emp_id'),
        how='inner'
    )
    .filter(F.col('m.name') == 'Tom')
    .select(
        F.col('e.name').alias('employee'),
        F.col('e.role'),
    )
    .orderBy('e.name')
)
print("Tom's direct reports:")
df_toms_team.show()

---
## 5. FULL OUTER JOIN — All Rows from Both Sides

In [ ]:
# Every customer + every order — NULLs on whichever side has no match
df_full = (
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .select('customer_id', 'name', 'order_id', 'amount')
    .orderBy('customer_id', 'order_id')
)
print('FULL OUTER JOIN — Dave (no orders) and orphan order both appear:')
df_full.show()

In [ ]:
# Only unmatched rows + reason label
df_unmatched = (
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .filter(
        F.col('order_id').isNull() | F.col('name').isNull()
    )
    .select(
        F.col('name').alias('customer'),
        'order_id',
        'amount',
        F.when(F.col('name').isNull(),     'Orphan order')
         .when(F.col('order_id').isNull(), 'No orders')
         .alias('reason')
    )
)
print('Unmatched rows from both sides:')
df_unmatched.show()

---
## 6. CROSS JOIN — Cartesian Product

In [ ]:
# Every customer × every product = 5 × 4 = 20 rows
df_catalog = (
    df_customers
    .crossJoin(df_products)
    .select(
        F.col('name').alias('customer'),
        'product_name',
        'category',
        'price',
    )
    .orderBy('name', 'product_name')
)
print(f'CROSS JOIN — {df_catalog.count()} rows (5 × 4):')
df_catalog.show()

---
## 7. Handling Ambiguous Column Names

In [ ]:
# When both DataFrames have the same column name (other than the join key)
# Use withColumnRenamed before join OR alias + F.col('alias.col') after

# Fix: rename before join
df_orders_renamed = df_orders.withColumnRenamed('status', 'order_status')
df_joined = df_orders_renamed.join(df_customers, on='customer_id', how='inner')
df_joined.select('order_id', 'name', 'order_status', 'amount').show(3)

---
## Quick Reference

| Join | PySpark | NULL behaviour |
|------|---------|----------------|
| INNER | `.join(df2, on, how='inner')` | Unmatched rows dropped from both sides |
| LEFT | `.join(df2, on, how='left')` | Right side NULL for unmatched |
| RIGHT | `.join(df2, on, how='right')` | Left side NULL for unmatched |
| FULL OUTER | `.join(df2, on, how='outer')` | NULL on whichever side has no match |
| SELF | `.alias('a').join(.alias('b'), F.col('a.x')==F.col('b.y'), how='left')` | Use LEFT so top of hierarchy appears |
| CROSS | `.crossJoin(df2)` | All rows × all rows — no condition |

In [ ]:
spark.stop()
print('Spark stopped.')